In [1]:
# Setup directory structure
import os

# Create directories
os.makedirs('my_model', exist_ok=True)
os.makedirs('my_model/code', exist_ok=True)

# Move model file
# Assuming your model.pth is in the current directory
if os.path.exists('model.pth'):
    os.rename('model.pth', 'my_model/model.pth')
    
print("done")

done


In [2]:
# Quick directory structure verification
def verify_structure():
    required_files = [
        'my_model/model.pth',
        'my_model/code/inference.py',
        'my_model/code/requirements.txt'
    ]
    
    for file_path in required_files:
        if not os.path.exists(file_path):
            print(f"Missing: {file_path}")
            return False
    return True

# Run verification
if verify_structure():
    print("Directory structure is correct")
else:
    print("Please fix the directory structure before proceeding")

Directory structure is correct


In [8]:
import tarfile
def create_model_archive():
    print("Creating model archive...")
    
    # Verify directory structure
    if not os.path.exists('my_model'):
        raise FileNotFoundError("my_model directory not found")
    if not os.path.exists('my_model/code'):
        raise FileNotFoundError("my_model/code directory not found")
    if not os.path.exists('my_model/model.pth'):
        raise FileNotFoundError("model.pth not found in my_model directory")
    if not os.path.exists('my_model/code/inference.py'):
        raise FileNotFoundError("inference.py not found in my_model/code directory")
    
    # Create archive
    with tarfile.open('model.tar.gz', 'w:gz') as tar:
        tar.add('my_model', arcname='.')
    
    # Verify archive contents
    with tarfile.open('model.tar.gz', 'r:gz') as tar:
        files = tar.getnames()
        print("\nArchive contents:")
        for f in files:
            print(f"- {f}")
    
    print("Model archive created successfully")

In [9]:
# Create model archive
create_model_archive()

Creating model archive...

Archive contents:
- .
- ./code
- ./code/inference.py
- ./code/requirements.txt
- ./model.pth
Model archive created successfully


In [14]:
import sagemaker
from sagemaker.pytorch import PyTorchModel
from sagemaker.serverless import ServerlessInferenceConfig
import boto3
from datetime import datetime

# Variables to set
TIMESTAMP = datetime.utcnow().strftime('%Y-%m-%d-%H-%M-%S')
REGION = 'ap-southeast-1'
ROLE_ARN = 'arn:aws:iam::522814726499:role/SageMaker-ExecutionRole-Scaneurysm'
USER = '522814726499'

def deploy_serverless_model():
    """Deploy model using serverless endpoint"""
    try:
        # Initialize sessions
        boto_session = boto3.Session(region_name=REGION)
        sagemaker_client = boto_session.client('sagemaker')
        s3_client = boto_session.client('s3')
        session = sagemaker.Session(
            boto_session=boto_session,
            sagemaker_client=sagemaker_client
        )
        bucket = session.default_bucket()

        print(f"\nStarting serverless deployment at {TIMESTAMP} UTC")
        print(f"User: {USER}")
        print(f"Role ARN: {ROLE_ARN}")
        print(f"Region: {REGION}")
        print(f"S3 Bucket: {bucket}")
        
        print("\nUploading model to S3...")
        model_key = f'models/brain-ct/model-{int(datetime.utcnow().timestamp())}.tar.gz'
        with open('model.tar.gz', 'rb') as f:
            s3_client.upload_fileobj(
                f,
                bucket,
                model_key,
                ExtraArgs={
                    'ServerSideEncryption': 'AES256'
                }
            )
        
        model_data = f"s3://{bucket}/{model_key}"
        print(f"Model uploaded to: {model_data}")

        # Create PyTorch model with updated configuration
        print("\nCreating PyTorch model...")
        model = PyTorchModel(
            model_data=model_data,
            role=ROLE_ARN,
            framework_version="2.1.0",
            py_version="py310",
            entry_point="inference.py",
            sagemaker_session=session,
            env={
                'SAGEMAKER_SUBMIT_DIRECTORY': '/opt/ml/model/code',
                'SAGEMAKER_PROGRAM': 'inference.py',
                'PYTHONPATH': '/opt/ml/model/code',
            }
        )

        # Configure serverless config
        serverless_config = ServerlessInferenceConfig(
            memory_size_in_mb=3072,
            max_concurrency=2
        )

        # Deploy model with serverless configuration
        endpoint_name = f"brain-ct-serverless-{int(datetime.utcnow().timestamp())}"
        print(f"\nDeploying serverless endpoint: {endpoint_name}")
        
        predictor = model.deploy(
            endpoint_name=endpoint_name,
            serverless_inference_config=serverless_config
        )

        # Wait for endpoint
        waiter = session.sagemaker_client.get_waiter('endpoint_in_service')
        print("\nWaiting for endpoint deployment...")
        waiter.wait(
            EndpointName=endpoint_name,
            WaiterConfig={
                'Delay': 30,
                'MaxAttempts': 20
            }
        )
        
        # Check status
        endpoint_status = session.sagemaker_client.describe_endpoint(
            EndpointName=endpoint_name
        )
        
        if endpoint_status['EndpointStatus'] == 'InService':
            print("\n✅ Serverless endpoint deployed successfully!")
            print(f"Status: {endpoint_status['EndpointStatus']}")
            print(f"Created: {endpoint_status['CreationTime']}")
            print(f"Endpoint Name: {endpoint_name}")
            print("\nConfiguration:")
            print("✓ PyTorch 1.13.1 with Python 3.9")
            print("✓ Serverless compute configured")
            print(f"✓ Memory size: 3072 MB")
            print(f"✓ Max concurrency: 20")
        else:
            print(f"\n⚠️ Endpoint status: {endpoint_status['EndpointStatus']}")
            
        return predictor
        
    except Exception as e:
        print(f"\n❌ Error during deployment: {str(e)}")
        raise

In [15]:
# Deploy model
predictor = deploy_serverless_model()

[04/28/25 20:16:26] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=770942;file:///Users/malcolmpaltiraja/Documents/Uni_Files/FT3162/AI-repo/Scaneurysm-AI-ML/sagemaker-env/lib/python3.9/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=12310;file:///Users/malcolmpaltiraja/Documents/Uni_Files/FT3162/AI-repo/Scaneurysm-AI-ML/sagemaker-env/lib/python3.9/site-packages/botocore/credentials.py#1352\1352]8;;\


Starting serverless deployment at 2025-04-28-12-16-24 UTC
User: 522814726499
Role ARN: arn:aws:iam::522814726499:role/SageMaker-ExecutionRole-Scaneurysm
Region: ap-southeast-1
S3 Bucket: sagemaker-ap-southeast-1-522814726499

Uploading model to S3...
Model uploaded to: s3://sagemaker-ap-southeast-1-522814726499/models/brain-ct/model-1745813787.tar.gz

Creating PyTorch model...

Deploying serverless endpoint: brain-ct-serverless-1745813813


[04/28/25 20:16:53] INFO     Defaulting to CPU type when using serverless inference               ]8;id=543673;file:///Users/malcolmpaltiraja/Documents/Uni_Files/FT3162/AI-repo/Scaneurysm-AI-ML/sagemaker-env/lib/python3.9/site-packages/sagemaker/image_uris.py\image_uris.py]8;;\:]8;id=513823;file:///Users/malcolmpaltiraja/Documents/Uni_Files/FT3162/AI-repo/Scaneurysm-AI-ML/sagemaker-env/lib/python3.9/site-packages/sagemaker/image_uris.py#538\538]8;;\

                    INFO     Repacking model artifact                                                  ]8;id=708348;file:///Users/malcolmpaltiraja/Documents/Uni_Files/FT3162/AI-repo/Scaneurysm-AI-ML/sagemaker-env/lib/python3.9/site-packages/sagemaker/model.py\model.py]8;;\:]8;id=264943;file:///Users/malcolmpaltiraja/Documents/Uni_Files/FT3162/AI-repo/Scaneurysm-AI-ML/sagemaker-env/lib/python3.9/site-packages/sagemaker/model.py#820\820]8;;\
                             (s3://sagemaker-ap-southeast-1-522814726499/models/brain-ct/model-1745813             
                             787.tar.gz), script artifact (None), and dependencies ([]) into single                
                             tar.gz file located at                                                                
                             s3://sagemaker-ap-southeast-1-522814726499/pytorch-inference-2025-04-28-1             
                             2-16-53-722/model.tar.gz. This may take some time depending on model                  
                             size...                                                                               

[04/28/25 20:18:15] INFO     Creating model with name: pytorch-inference-2025-04-28-12-18-15-481    ]8;id=574081;file:///Users/malcolmpaltiraja/Documents/Uni_Files/FT3162/AI-repo/Scaneurysm-AI-ML/sagemaker-env/lib/python3.9/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=325850;file:///Users/malcolmpaltiraja/Documents/Uni_Files/FT3162/AI-repo/Scaneurysm-AI-ML/sagemaker-env/lib/python3.9/site-packages/sagemaker/session.py#4094\4094]8;;\

[04/28/25 20:18:16] INFO     Creating endpoint-config with name brain-ct-serverless-1745813813      ]8;id=333575;file:///Users/malcolmpaltiraja/Documents/Uni_Files/FT3162/AI-repo/Scaneurysm-AI-ML/sagemaker-env/lib/python3.9/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=804999;file:///Users/malcolmpaltiraja/Documents/Uni_Files/FT3162/AI-repo/Scaneurysm-AI-ML/sagemaker-env/lib/python3.9/site-packages/sagemaker/session.py#6019\6019]8;;\

                    INFO     Creating endpoint with name brain-ct-serverless-1745813813             ]8;id=164276;file:///Users/malcolmpaltiraja/Documents/Uni_Files/FT3162/AI-repo/Scaneurysm-AI-ML/sagemaker-env/lib/python3.9/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=739752;file:///Users/malcolmpaltiraja/Documents/Uni_Files/FT3162/AI-repo/Scaneurysm-AI-ML/sagemaker-env/lib/python3.9/site-packages/sagemaker/session.py#4841\4841]8;;\

----!
Waiting for endpoint deployment...

✅ Serverless endpoint deployed successfully!
Status: InService
Created: 2025-04-28 20:18:16.860000+08:00
Endpoint Name: brain-ct-serverless-1745813813

Configuration:
✓ PyTorch 1.13.1 with Python 3.9
✓ Serverless compute configured
✓ Memory size: 3072 MB
✓ Max concurrency: 20


In [ ]:
import boto3
import json

def invoke_endpoint(endpoint_name, image_url):
    runtime = boto3.client('sagemaker-runtime')
    
    # Prepare the input
    input_data = {
        "url": image_url
    }
    
    # Invoke endpoint
    response = runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType='application/json',
        Body=json.dumps(input_data)
    )
    
    # Parse response
    result = json.loads(response['Body'].read().decode())
    return result

In [ ]:

res = invoke_endpoint('brain-ct-serverless-1745728805','https://upload.wikimedia.org/wikipedia/commons/b/b2/MRI_of_Human_Brain.jpg')

print(res)

In [22]:
!pip install shap matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.1/547.1 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 31.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.8/28.8 MB 19.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 42.8 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.3/30.3 MB 50.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15/15 [shap]2m14/15 [shap]otlib]n]ources]


In [23]:
import boto3
import json
import numpy as np
import matplotlib.pyplot as plt
import shap

def invoke_endpoint_with_shap(endpoint_name, image_url):
    """
    Invoke endpoint and visualize SHAP values
    Args:
        endpoint_name (str): SageMaker endpoint name
        image_url (str): URL of the image to analyze
    """
    runtime = boto3.client('sagemaker-runtime')
    
    # Prepare input with SHAP flag enabled
    input_data = {
        "url": image_url,
        "calculate_shap": True
    }
    
    # Invoke endpoint
    response = runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType='application/json',
        Body=json.dumps(input_data)
    )
    
    # Parse response
    result = json.loads(response['Body'].read().decode())
    
    # Print prediction results
    print("\nPrediction Results:")
    print(f"Prediction: {result['prediction']}")
    print(f"Confidence: {result['confidence']:.2f}")
    print("\nProbabilities:")
    print(f"Non-aneurysm: {result['probabilities']['non_aneurysm']:.2f}")
    print(f"Aneurysm: {result['probabilities']['aneurysm']:.2f}")
    
    # Handle SHAP values if present
    if 'shap_values' in result:
        print("\nVisualizing SHAP values...")
        
        # Convert SHAP values to numpy arrays
        shap_values = [
            np.array(result['shap_values']['non_aneurysm']),
            np.array(result['shap_values']['aneurysm'])
        ]
        
        # Plot SHAP values
        plt.figure(figsize=(10, 5))
        
        # Plot for non-aneurysm class
        plt.subplot(1, 2, 1)
        shap.image_plot(shap_values[0], show=False)
        plt.title('SHAP values for Non-aneurysm')
        
        # Plot for aneurysm class
        plt.subplot(1, 2, 2)
        shap.image_plot(shap_values[1], show=False)
        plt.title('SHAP values for Aneurysm')
        
        plt.tight_layout()
        plt.show()
    
    if 'shap_error' in result:
        print(f"\nSHAP Error: {result['shap_error']}")
    
    return result

# Example usage
response = invoke_endpoint_with_shap(
    endpoint_name='brain-ct-serverless-1745813813',
    image_url='https://prod-images-static.radiopaedia.org/images/61513769/9131de5120872bcc1ad19d8198c759904a4b78efa405550c86ec0f19d3dda590_big_gallery.jpeg'
)

[04/28/25 20:28:25] WARNING  Matplotlib is building the font cache; this may take a moment.    ]8;id=742837;file:///Users/malcolmpaltiraja/Documents/Uni_Files/FT3162/AI-repo/Scaneurysm-AI-ML/sagemaker-env/lib/python3.9/site-packages/matplotlib/font_manager.py\font_manager.py]8;;\:]8;id=561431;file:///Users/malcolmpaltiraja/Documents/Uni_Files/FT3162/AI-repo/Scaneurysm-AI-ML/sagemaker-env/lib/python3.9/site-packages/matplotlib/font_manager.py#1040\1040]8;;\

[04/28/25 20:28:34] INFO     Failed to extract font properties from                            ]8;id=704577;file:///Users/malcolmpaltiraja/Documents/Uni_Files/FT3162/AI-repo/Scaneurysm-AI-ML/sagemaker-env/lib/python3.9/site-packages/matplotlib/font_manager.py\font_manager.py]8;;\:]8;id=479889;file:///Users/malcolmpaltiraja/Documents/Uni_Files/FT3162/AI-repo/Scaneurysm-AI-ML/sagemaker-env/lib/python3.9/site-packages/matplotlib/font_manager.py#1052\1052]8;;\
                             /System/Library/PrivateFrameworks/FontServices.framework/Resource                     
                             s/Reserved/PingFangUI.ttc: In FT2Font: Can not load face                              
                             (locations (loca) table missing; error code 0x90)                                     

                    INFO     Failed to extract font properties from                            ]8;id=795083;file:///Users/malcolmpaltiraja/Documents/Uni_Files/FT3162/AI-repo/Scaneurysm-AI-ML/sagemaker-env/lib/python3.9/site-packages/matplotlib/font_manager.py\font_manager.py]8;;\:]8;id=759761;file:///Users/malcolmpaltiraja/Documents/Uni_Files/FT3162/AI-repo/Scaneurysm-AI-ML/sagemaker-env/lib/python3.9/site-packages/matplotlib/font_manager.py#1052\1052]8;;\
                             /System/Library/Fonts/Supplemental/NISC18030.ttf: In FT2Font:                         
                             Could not set the fontsize (invalid pixel size; error code 0x17)                      

                    INFO     Failed to extract font properties from                            ]8;id=70959;file:///Users/malcolmpaltiraja/Documents/Uni_Files/FT3162/AI-repo/Scaneurysm-AI-ML/sagemaker-env/lib/python3.9/site-packages/matplotlib/font_manager.py\font_manager.py]8;;\:]8;id=122864;file:///Users/malcolmpaltiraja/Documents/Uni_Files/FT3162/AI-repo/Scaneurysm-AI-ML/sagemaker-env/lib/python3.9/site-packages/matplotlib/font_manager.py#1052\1052]8;;\
                             /System/Library/Fonts/LastResort.otf: tuple indices must be                           
                             integers or slices, not str                                                           

                    INFO     Failed to extract font properties from                            ]8;id=727722;file:///Users/malcolmpaltiraja/Documents/Uni_Files/FT3162/AI-repo/Scaneurysm-AI-ML/sagemaker-env/lib/python3.9/site-packages/matplotlib/font_manager.py\font_manager.py]8;;\:]8;id=117093;file:///Users/malcolmpaltiraja/Documents/Uni_Files/FT3162/AI-repo/Scaneurysm-AI-ML/sagemaker-env/lib/python3.9/site-packages/matplotlib/font_manager.py#1052\1052]8;;\
                             /System/Library/Fonts/Apple Color Emoji.ttc: In FT2Font: Could                        
                             not set the fontsize (invalid pixel size; error code 0x17)                            

                    INFO     generated new fontManager                                         ]8;id=876410;file:///Users/malcolmpaltiraja/Documents/Uni_Files/FT3162/AI-repo/Scaneurysm-AI-ML/sagemaker-env/lib/python3.9/site-packages/matplotlib/font_manager.py\font_manager.py]8;;\:]8;id=439106;file:///Users/malcolmpaltiraja/Documents/Uni_Files/FT3162/AI-repo/Scaneurysm-AI-ML/sagemaker-env/lib/python3.9/site-packages/matplotlib/font_manager.py#1584\1584]8;;\


Prediction Results:
Prediction: Non-aneurysm
Confidence: 0.62

Probabilities:
Non-aneurysm: 0.62
Aneurysm: 0.38


In [2]:
import boto3
import datetime
import json
from typing import Dict, Any

def upload_baseline_to_s3(baseline_data: Dict) -> str:
    """Upload baseline data to S3 and return the S3 URI"""
    s3_client = boto3.client('s3', region_name='ap-southeast-1')
    bucket_name = 'brain-ct-async-inference-krooldonutz'
    timestamp = datetime.datetime.now().strftime('%Y%m%d%H%M%S')
    key = f'clarify-baseline/baseline-{timestamp}.json'
    
    try:
        s3_client.put_object(
            Bucket=bucket_name,
            Key=key,
            Body=json.dumps(baseline_data)
        )
        return f's3://{bucket_name}/{key}'
    except Exception as e:
        raise ValueError(f"Error uploading baseline to S3: {str(e)}")

def create_clarify_config(
    model_name: str,
    instance_type: str = 'ml.t2.medium',  # Using a cost-effective instance type
    instance_count: int = 1
) -> Dict[str, Any]:
    timestamp = datetime.datetime.now().strftime('%Y%m%d%H%M%S')
    config_name = f'brain-ct-clarify-{timestamp}'
    
    # Create baseline data
    baseline_data = {
        "url": "",
        "pixel_values": [0.0] * (32 * 32)  # Smaller baseline for testing
    }
    
    # Upload baseline to S3
    baseline_s3_uri = upload_baseline_to_s3(baseline_data)
    
    config = {
        "EndpointConfigName": config_name,
        "ProductionVariants": [
            {
                "VariantName": "AllTraffic",
                "ModelName": model_name,
                "InstanceType": instance_type,
                "InitialInstanceCount": instance_count,
                "InitialVariantWeight": 1.0
            }
        ],
        "ExplainerConfig": {
            "ClarifyExplainerConfig": {
                "EnableExplanations": "true",
                "InferenceConfig": {
                    "MaxPayloadInMB": 6,
                    "MaxRecordCount": 1
                },
                "ShapConfig": {
                    "NumberOfSamples": 100,
                    "Seed": 123,
                    "ShapBaselineConfig": {
                        "ShapBaselineUri": baseline_s3_uri,
                        "MimeType": "application/json"
                    }
                }
            }
        }
    }
    
    return config

def deploy_clarify_endpoint(
    model_name: str = 'pytorch-inference-2025-04-28-12-18-15-481',
    instance_type: str = 'ml.t2.medium',
    instance_count: int = 1
):
    sagemaker_client = boto3.client('sagemaker', region_name='ap-southeast-1')
    
    try:
        # Create endpoint configuration
        config = create_clarify_config(
            model_name=model_name,
            instance_type=instance_type,
            instance_count=instance_count
        )
        
        # Print config for debugging
        print("Creating endpoint config with configuration:")
        print(json.dumps(config, indent=2))
        
        # Create endpoint config
        response = sagemaker_client.create_endpoint_config(**config)
        config_name = config["EndpointConfigName"]
        print(f"Created endpoint config: {config_name}")
        
        # Create endpoint
        timestamp = datetime.datetime.now().strftime('%Y%m%d%H%M%S')
        endpoint_name = f'brain-ct-clarify-{timestamp}'
        
        response = sagemaker_client.create_endpoint(
            EndpointName=endpoint_name,
            EndpointConfigName=config_name
        )
        
        print(f"Creating endpoint with Clarify: {endpoint_name}")
        return endpoint_name
        
    except Exception as e:
        print(f"Error deploying Clarify endpoint: {str(e)}")
        raise

# Deploy the endpoint
try:
    new_endpoint_name = deploy_clarify_endpoint(
        model_name='pytorch-inference-2025-04-28-12-18-15-481',
        instance_type='ml.t2.medium',  # You can adjust this based on your needs
        instance_count=1
    )
    print(f"New endpoint being created: {new_endpoint_name}")
except Exception as e:
    print(f"Detailed error: {str(e)}")

Creating endpoint config with configuration:
{
  "EndpointConfigName": "brain-ct-clarify-20250501001401",
  "ProductionVariants": [
    {
      "VariantName": "AllTraffic",
      "ModelName": "pytorch-inference-2025-04-28-12-18-15-481",
      "InstanceType": "ml.t2.medium",
      "InitialInstanceCount": 1,
      "InitialVariantWeight": 1.0
    }
  ],
  "ExplainerConfig": {
    "ClarifyExplainerConfig": {
      "EnableExplanations": "true",
      "InferenceConfig": {
        "MaxPayloadInMB": 6,
        "MaxRecordCount": 1
      },
      "ShapConfig": {
        "NumberOfSamples": 100,
        "Seed": 123,
        "ShapBaselineConfig": {
          "ShapBaselineUri": "s3://brain-ct-async-inference-krooldonutz/clarify-baseline/baseline-20250501001401.json",
          "MimeType": "application/json"
        }
      }
    }
  }
}
Created endpoint config: brain-ct-clarify-20250501001401
Creating endpoint with Clarify: brain-ct-clarify-20250501001402
New endpoint being created: brain-ct-clarify

In [ ]:
from time import time, sleep
import boto3
import datetime
import json
from typing import Dict, Any

def create_clarify_config(
    model_name: str,
    instance_type: str = 'ml.t2.medium',
    instance_count: int = 1
) -> Dict[str, Any]:
    timestamp = datetime.datetime.now().strftime('%Y%m%d%H%M%S')
    config_name = f'brain-ct-clarify-{timestamp}'
    
    # Create baseline data matching your model's exact output structure
    baseline_data = {
        "url": "https://prod-images-static.radiopaedia.org/images/61513769/9131de5120872bcc1ad19d8198c759904a4b78efa405550c86ec0f19d3dda590_big_gallery.jpeg",
        "explain": True,
        "features": [[0.0] * 1024],
        "probabilities": {
            "non_aneurysm": 0.5,
            "aneurysm": 0.5
        }
    }
    
    # Upload baseline to S3
    baseline_s3_uri = upload_baseline_to_s3(baseline_data)
    
    config = {
        "EndpointConfigName": config_name,
        "ProductionVariants": [
            {
                "VariantName": "AllTraffic",
                "ModelName": model_name,
                "InstanceType": instance_type,
                "InitialInstanceCount": instance_count,
                "InitialVariantWeight": 1.0
            }
        ],
        "ExplainerConfig": {
            "ClarifyExplainerConfig": {
                "EnableExplanations": "true",
                "InferenceConfig": {
                    "MaxPayloadInMB": 10,
                    "MaxRecordCount": 1,
                    # "FeatureTypes": ["numerical"],
                    "FeaturesAttribute": "features",
                    "ProbabilityAttribute": "probabilities.non_aneurysm"  # Using the exact path to probability
                },
                "ShapConfig": {
                    "NumberOfSamples": 50,
                    "Seed": 123,
                    "ShapBaselineConfig": {
                        "ShapBaselineUri": baseline_s3_uri,
                        "MimeType": "application/json"
                    }
                }
            }
        }
    }
    
    return config


def deploy_clarify_endpoint(
    model_name: str = 'pytorch-inference-2025-04-28-12-18-15-481',
    instance_type: str = 'ml.t2.medium',
    instance_count: int = 1
):
    sagemaker_client = boto3.client('sagemaker', region_name='ap-southeast-1')
    
    try:
        # Create endpoint configuration
        config = create_clarify_config(
            model_name=model_name,
            instance_type=instance_type,
            instance_count=instance_count
        )
        
        # Print config for debugging
        print("Creating endpoint config with configuration:")
        print(json.dumps(config, indent=2))
        
        # Create endpoint config
        response = sagemaker_client.create_endpoint_config(**config)
        config_name = config["EndpointConfigName"]
        print(f"Created endpoint config: {config_name}")
        
        # Create endpoint
        timestamp = datetime.datetime.now().strftime('%Y%m%d%H%M%S')
        endpoint_name = f'brain-ct-clarify-{timestamp}'
        
        response = sagemaker_client.create_endpoint(
            EndpointName=endpoint_name,
            EndpointConfigName=config_name
        )
        
        print(f"Creating endpoint with Clarify: {endpoint_name}")
        return endpoint_name
        
    except Exception as e:
        print(f"Error deploying Clarify endpoint: {str(e)}")
        raise

def wait_for_endpoint(endpoint_name: str, timeout: int = 900):
    """Wait for endpoint to be InService"""
    sagemaker_client = boto3.client('sagemaker', region_name='ap-southeast-1')
    start_time = datetime.datetime.now()
    
    print(f"Waiting for endpoint {endpoint_name} to be ready...")
    
    while True:
        try:
            response = sagemaker_client.describe_endpoint(EndpointName=endpoint_name)
            status = response['EndpointStatus']
            
            if status == 'InService':
                print(f"Endpoint is ready!")
                return True
            elif status == 'Failed':
                failure_reason = response.get('FailureReason', 'No failure reason provided')
                print(f"Endpoint creation failed: {failure_reason}")
                return False
                
            elapsed_time = (datetime.datetime.now() - start_time).total_seconds()
            if elapsed_time > timeout:
                print(f"Timeout waiting for endpoint")
                return False
                
            print(f"Current status: {status}. Waiting...")
            sleep(30)
            
        except Exception as e:
            print(f"Error checking endpoint status: {str(e)}")
            return False

# # Main execution
if __name__ == "__main__":
    try:
        print(f"Starting deployment at: {datetime.datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S UTC')}")
        
        # Deploy new endpoint
        new_endpoint_name = deploy_clarify_endpoint(
            model_name='pytorch-inference-2025-04-28-12-18-15-481',
            instance_type='ml.t2.medium',
            instance_count=1
        )
        print(f"New endpoint being created: {new_endpoint_name}")
        
        # Wait for endpoint to be ready
        if wait_for_endpoint(new_endpoint_name):
            print("Endpoint is ready for testing!")
            
            # Test the endpoint
            result = test_endpoint(new_endpoint_name)
            print(f"Response: {json.dumps(result, indent=2)}")
            
        else:
            print("Endpoint creation failed or timed out")
            
    except Exception as e:
        print(f"Error in main execution: {str(e)}")

Starting deployment at: 2025-04-30 16:14:38 UTC
Creating endpoint config with configuration:
{
  "EndpointConfigName": "brain-ct-clarify-20250501001438",
  "ProductionVariants": [
    {
      "VariantName": "AllTraffic",
      "ModelName": "pytorch-inference-2025-04-28-12-18-15-481",
      "InstanceType": "ml.t2.medium",
      "InitialInstanceCount": 1,
      "InitialVariantWeight": 1.0
    }
  ],
  "ExplainerConfig": {
    "ClarifyExplainerConfig": {
      "EnableExplanations": "true",
      "InferenceConfig": {
        "MaxPayloadInMB": 10,
        "MaxRecordCount": 1,
        "FeaturesAttribute": "features",
        "ProbabilityAttribute": "probabilities.non_aneurysm"
      },
      "ShapConfig": {
        "NumberOfSamples": 50,
        "Seed": 123,
        "ShapBaselineConfig": {
          "ShapBaselineUri": "s3://brain-ct-async-inference-krooldonutz/clarify-baseline/baseline-20250501001438.json",
          "MimeType": "application/json"
        }
      }
    }
  }
}
Created endpoi

In [ ]:
def test_endpoint(endpoint_name: str):
    """
    Test the SageMaker endpoint with a sample input matching the baseline format
    """
    runtime_client = boto3.client('runtime.sagemaker', region_name='ap-southeast-1')
    
    # Construct the test payload - matching the baseline format defined in create_clarify_config
    test_payload = {
        "url": "https://prod-images-static.radiopaedia.org/images/61513769/9131de5120872bcc1ad19d8198c759904a4b78efa405550c86ec0f19d3dda590_big_gallery.jpeg",
        "explain": True,
        "features": [[0.0] * 1024],  # Adding the features array as defined in baseline
        "probabilities": {
            "non_aneurysm": 0.5,
            "aneurysm": 0.5
        }
    }
    
    try:
        print(f"Testing inference at: {datetime.datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S UTC')}")
        
        response = runtime_client.invoke_endpoint(
            EndpointName=endpoint_name,
            ContentType='application/json',
            Body=json.dumps(test_payload)
        )
        
        result = json.loads(response['Body'].read().decode())
        print("Inference successful!")
        return result
        
    except Exception as e:
        print(f"Error testing endpoint: {str(e)}")
        if hasattr(e, 'response'):
            print(f"Error response: {e.response}")
            if 'ResponseMetadata' in e.response:
                print(f"Request ID: {e.response['ResponseMetadata'].get('RequestId')}")
        raise
# Example usage:
result = test_endpoint('brain-ct-clarify-20250501001438')

Testing inference at: 2025-04-30 16:36:08 UTC
Sending payload: {
  "url": "https://prod-images-static.radiopaedia.org/images/61513769/9131de5120872bcc1ad19d8198c759904a4b78efa405550c86ec0f19d3dda590_big_gallery.jpeg",
  "explain": true,
  "features": [
    [
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
    

InternalFailure: An error occurred (InternalFailure) when calling the InvokeEndpoint operation (reached max retries: 4): An exception occurred while sending request to model. Please contact customer support regarding request e2087ba9-79ae-4f9d-8090-059fdcba6ef3.

In [6]:
import boto3
client = boto3.client('sagemaker')
response = client.describe_endpoint(EndpointName='brain-ct-clarify-20250501001438')
print(response['EndpointStatus'])

InService
